# agent_v2 on Kaggle

**Notebook settings:** GPU T4 x2, Internet on, the private dataset `vladyslavsh0/chimera-agent-challenge-task-1` added as input, and the Kaggle secret `HF_TOKEN` (read token with access to `google/embeddinggemma-300m`) attached.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!find /kaggle/input -maxdepth 6 -type d -name agent_input

Dataset path, commit and run name of the dev run reported in `results/agent_v2.md`.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

DATA_ROOT = "/kaggle/input/datasets/vladyslavsh0/chimera-agent-challenge-task-1"
COMMIT = "1b6b70630ea4bd0c510e55d086bb0bc25d28e5a0"
SYSTEM = "agent_v2"
RUN = "agent_v2_dev"
REPO = "/kaggle/working/CHIMERA-Agent"
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
![ -d {REPO} ] || git clone -q https://github.com/VlaDisLav23232/CHIMERA-Agent.git {REPO}
%cd {REPO}
!git checkout -q {COMMIT} && git log -1 --format='%H %s'

Python 3.12 environment, the pinned baseline and the agent dependencies.

In [ ]:
!pip install -q uv
!uv venv -q --python 3.12 .venv
!uv pip install -q --python .venv/bin/python -r requirements.txt
!bash scripts/setup_agent.sh

Ollama with the context length the runner checks, then the models (Gemma via Ollama, EmbeddingGemma via Hugging Face).

In [ ]:
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
ollama = subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_CONTEXT_LENGTH": "16384"},
                          stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama --version

In [ ]:
!bash scripts/setup_agent.sh --models
!ollama list

Unit tests on Linux. The evaluation tests need the organizers' evaluator; tests that need local data skip here.

In [ ]:
!bash scripts/fetch_evaluator.sh
!.venv/bin/python -m unittest discover -s tests -t .

A config copy that points `data_dir` at the dataset's agent inputs. Everything else stays as in `configs/agent.json`.

In [ ]:
import json
config = json.load(open("configs/agent.json"))
config["data_dir"] = f"{DATA_ROOT}/task1/agent_input"
json.dump(config, open("/kaggle/working/agent_kaggle.json", "w"), indent=2)
print(config)

Full dev run, 60 cases. The reported run took 103 min; 60 of 60 cases succeeded.

In [ ]:
!.venv/bin/python -m chimera_agent.run --name {RUN} --system {SYSTEM} --all-dev --config /kaggle/working/agent_kaggle.json

Run summary: counts only, no patient text.

In [ ]:
import collections, glob
info = json.load(open(f"results/{RUN}/run_info.json"))
print(info["status"], collections.Counter(r["status"] for r in info["cases"].values()))
print({c[-6:]: (r["status"], r.get("error_type")) for c, r in info["cases"].items() if r["status"] != "succeeded"})
traces = [json.load(open(p)) for p in glob.glob(f"results/{RUN}/trace/*.json")]
print("escape repairs:", sum(t["escape_repairs"] for t in traces), "in", sum(t["escape_repairs"] > 0 for t in traces), "cases")
!.venv/bin/python -m evaluation.audit checks results/{RUN}

Zip a run without its index copy and download it right away (`/kaggle/working` is lost when the session ends). The zip contains patient information: keep it local, never commit or share it.

In [ ]:
!cd results && zip -qr /kaggle/working/{RUN}.zip {RUN} -x "{RUN}/resources/*" && ls -lh /kaggle/working/{RUN}.zip